In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import MinMaxScaler

In [2]:
df = pd.read_csv('Parkison_Dataset.csv')

In [3]:
df.shape

(756, 755)

In [4]:
df.head()

,id,gender,PPE,DFA,RPDE,numPulses,numPeriodsPulses,meanPeriodPulses,stdDevPeriodPulses,locPctJitter,...,tqwt_kurtosisValue_dec_28,tqwt_kurtosisValue_dec_29,tqwt_kurtosisValue_dec_30,tqwt_kurtosisValue_dec_31,tqwt_kurtosisValue_dec_32,tqwt_kurtosisValue_dec_33,tqwt_kurtosisValue_dec_34,tqwt_kurtosisValue_dec_35,tqwt_kurtosisValue_dec_36,class
0,0,1,0.85247,0.71826,0.57227,240,239,0.008064,0.000087,0.00218,...,1.5620,2.6445,3.8686,4.2105,5.1221,4.4625,2.6202,3.0004,18.9405,1
1,0,1,0.76686,0.69481,0.53966,234,233,0.008258,0.000073,0.00195,...,1.5589,3.6107,23.5155,14.1962,11.0261,9.5082,6.5245,6.3431,45.1780,1
2,0,1,0.85083,0.67604,0.58982,232,231,0.008340,0.000060,0.00176,...,1.5643,2.3308,9.4959,10.7458,11.0177,4.8066,2.9199,3.1495,4.7666,1
3,1,0,0.41121,0.79672,0.59257,178,177,0.010858,0.000183,0.00419,...,3.7805,3.5664,5.2558,14.0403,4.2235,4.6857,4.8460,6.2650,4.0603,1
4,1,0,0.32790,0.79782,0.53028,236,235,0.008162,0.002669,0.00535,...,6.1727,5.8416,6.0805,5.7621,7.7817,11.6891,8.2103,5.0559,6.1164,1


In [5]:
df.id.nunique()

252

In [6]:
print('Summary of attribute datatypes:\n', df.dtypes.value_counts(), sep="")

Summary of attribute datatypes:
float64    749
int64        6
dtype: int64


In [7]:
print("Number of null values:", df.isnull().sum().sum())

Number of null values: 0


### Data Preprocessing

In [9]:
from sklearn.model_selection import train_test_split

In [10]:
X_train_origin, X_test_origin, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=91)

NameError: name 'X' is not defined

In [11]:
from sklearn.preprocessing import StandardScaler

In [13]:
X_train_origin.head()

NameError: name 'X_train_origin' is not defined

In [ ]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
scaler = MinMaxScaler()
scaler.fit(X_train_origin)
X_train = pd.DataFrame(scaler.transform(X_train_origin))
X_train.columns = X_train_origin.columns
X_train['gender'] = X_train['gender'].astype('category')
X_train.head()

In [ ]:
X_test = pd.DataFrame(scaler.transform(X_test_origin[X_test_origin.columns]))
X_test.columns = X_test_origin.columns
X_test['gender'] = X_test['gender'].astype('category')
X_test.head()

In [ ]:
y_train.value_counts()

In [ ]:
y_test.value_counts()

### Problem Definition
**Input:** $X \in \mathbb{R}^{d}, d = 243$

**Output:** $Y = \{0, 1\}$

**Classifier:** Calssification uses a function $f$ (called a classifier) to map input $x$ to class $y$. 

$y = f(x) : f$ takes in $x \in X$ and declares its class to be $y \in Y$

## II. Modeling

### 1. k-Nearest Neighbors Classifier (kNN)
KNNs classify the unseen instance based on the K points in the training set which are nearest to it. It is a **non-parametric method**. 

#### Non-parametric Models
Non-parametric models differ from parametric models in that the model structure is not specified a priori but is instead determined from data. The term non-parametric is not meant to imply that such models completely lack parameters but that the number and nature of the parameters are flexible and not fixed in advance.

Source: https://en.wikipedia.org/wiki/Nonparametric_statistics#Non-parametric_models

#### Algorithm
Given data $(x_1, y_1),...,(x_n, y_n)$, construct the $k$-NN classifier as follows:
For a new input $s$,
1. Return the $k$ points closest to $x$, indexed as $x_{i_1},...x_{i_k}$.
2. Return the majority-vote of $y_{i_1}, y_{i_2},..., y_{i_k}$.
The default distance for data in $\mathbb{R}^d$ is the Euclidean one:
$$\|u-v\|_2 = \big(\sum_{i=1}^{d}(u_i-v_i)^2\big)^\frac{1}{2}$$

In [14]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score

In [15]:
param_grid = {'n_neighbors': np.arange(3, 13)}
gs_kNN = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5)
gs_kNN.fit(X_train, y_train)
print("Best Number of Neighbors:", gs_kNN.best_params_)
print("Accuracy on Training Set:", gs_kNN.best_score_)

y_pred_prob = gs_kNN.predict_proba(X_test)[:,1]
print("Accuracy on Test Set:", gs_kNN.score(X_test, y_test))
print("AUC:", roc_auc_score(y_test, y_pred_prob))

NameError: name 'X_train' is not defined

#### Observe how the accuracy change as k grows
Note: The accuracy is different from the results from grid search because the whole training set is used to fit the model.

In [ ]:
# Compute training and test errors by k
training_error = list()
test_error = list()
for k in np.arange(3, 13):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    training_error.append(knn.score(X_train, y_train))
    test_error.append(knn.score(X_test, y_test))

In [ ]:
plt.plot(training_error, label = 'Training set accuracy')
plt.plot(test_error, label = 'Test set accuracy')
plt.legend()
plt.xticks(np.arange(0, 10), np.arange(3, 13))
plt.show()

### 2. Naive Bayes 

With Naive Bayes classifier we predict the class of a new $x$ to be the most probable lable given model and training data $(x_1, y_1), ..., (x_n, y_n)$.

#### Bayes Classifier
Before talking about the algorithm of Naive Bayes, we need to know **Bayes Classifer**:

$$f(x) = \operatorname*{arg\,max}_{y \in Y} P(Y = y| X = x)$$

For a particular input $x$, predict the label to be the most probable label conditioned on $x$ according to the true underlying distribution given to us from nature.

#### Bayes Rule: 
$$p(y|x) = \frac{p(x|y)p(y)}{p(x)}$$

From Bayes rule we equivalently have
$$ f(x) \approx \operatorname*{arg\,max}_{y \in Y} P(Y = y) \times P(X = x | Y = y) $$
- $P(Y = y)$ is called the $\textit{class prior}$.
- $P(X = x|Y = y)$ is called the $\textit{class conditional distribution}$ of X.
- In practice we don't know either of these, so we approximate them.

Aside: If $X$ is a continuous-valued random variable, replace $P(X = x|Y = y)$ with class conditional density $p(x|Y=y)$.

Problem: We can't construct the Bayes classifier without knowing $P(Y = y|X = x)$, or equv., $P(X = x|Y = y)$ and $P(Y = y)$. All we have are labeled examples drowm from the distribution.

#### Naive Bayes Algorithm
We have to $\textit{define } p(X = x|Y = y)$.

Naive Bayes is a Bayes classifier that makes the assumption
$$p(X = x|Y = y) = \prod_{j=1}^{d}p_j(x(j)|Y = y),$$
i.e., it treats the dimension of $X$ as $\textit{conditionally independent}$ given $y$. 

Note: Each dimension might not be independent with each other, but they are conditionally independent given y.

#### Discriminative Model vs. Generative Model

Bayes Classifer and Naive Bayes are ***generative models***. Unlike distriminative models, generative models consider the joint probability distribution on $X \times y, P(X, y)$, make the prediction on the probability of each lable $y$ given $X$, $p(y|X = x)$, and then **pick the most likely lable $y$**.

##### Discriminative Algorithms
- Idea: model $p(y|x)$, conditional distribution of $y$ given $x$.
- In Discriminative Algorithms: find a **decision boundary** that separates positive from negative example.
- To predict a new example, check on which side of the decision boundary it falls.
- **Model $p(y|x)$ directly**

##### Generative Algorithms
- Idea: Build a model for what positive examples look like and build a different model for what negative example look like.
- To predict a new example, match it with each of the models and see which match is best.
- Model $p(x|y)$ and $p(y)$!
- **Use Bayes rule to obtain $p(y|x) = \frac{p(x|y)p(y)}{p(x)}$**

The definition of generative and distriminative models:
- a generative model is a model of the conditional probability of the observable $X$, given a target $y$, symbolically, $P(X|Y=y)$
- a discriminative model is a model of the conditional probability of the target $Y$, given an observation $x$, symbolically, $P(Y|X=x)$

source: https://en.wikipedia.org/wiki/Generative_model#Definition



In [ ]:
from sklearn.naive_bayes import GaussianNB

In [ ]:
gnb = GaussianNB()
gnb.fit(X_train, y_train)
y_pred_prob = gnb.predict_proba(X_test)[:,1]

print("Accuracy on training set:", gnb.score(X_train, y_train))
print("Accuracy on test set:", gnb.score(X_test, y_test))
print("AUC:", roc_auc_score(y_test, y_pred_prob))

### 3. Logistic Regression
Let $(x_1, y_1),...,(x_n, y_n)$ be a set of binary labeled data with $y \in {-1, +1}$. $\textit{Logistic regression}$ models each $y_i$ as independently generated, with

$$P(y_i = +1|x_i, w) = \sigma(x_i^Tw), \ \ \sigma(x_i; w) = \frac{e^{x_iTw}}{1 + e^{x_iTw}}.$$

#### Sigmoid Function
From **Linear Discriminative Analysis**, we can directly plug in the **hyperplane representation** for the **log odds**. ***(See Appendix A: Concepts for Logistic Regression & Appendix B: Linear Classifiers)***

$$\ln\frac{p(y = +1|x)}{p(y = -1|x)} = x^Tw + w_0$$

Note: No restrictions on $w$ and $w_0$ compared to LDA.

Setting $p(y = -1|x) = 1 - p(y = -1|x)$, solve for $p(y = +1|x)$ to find 
$$ p(y = +1|x) = \frac{exp^{x^Tw + w_0}}{1 + exp^{x^Tw + w_0}} = \sigma(x^tw + w_0). $$

- This is called the sigmoid function
- We have chosen $x^Tw + w_0$ as the $\textit{link function}$ for log odds.
- If $x^Tw > 0$, then $\sigma(x^Tw) > 1/2$ and predict $y = +1$, and vice versa.
- We now get **a confidence in our prediction** via the probability of $\sigma (x^Tw)$.


#### Maximum Likelihood Estimation

Define $\sigma_i(w) = \sigma(x_i^Tw)$. The joint likihood of $y_1,...y_n$ is
$$p(y_1,...,y_n|x_1,...,x_n, w) = \prod_{i=1}^{n}p(y_i|x_i, w)
= \prod_{i=1}^{n}\sigma_i(w)^{\mathbb{I}(y_i=+1)}(1-\sigma_i(w))^{\mathbb{I}(y_i=-1)} 
= \prod_{i=1}^{n}\sigma_i(y_i \cdot w)$$

Note: here y = {+1, -1}

we want to maximize this over $w$.

The maximum likelihood solution for $w$ can be written 
$$ w_{ML} = \operatorname*{arg\,max}_{w} \sum_{i=1}^{n}\ln\sigma_i(y_i\cdot w)
= \operatorname*{arg\,max}_{w} L $$

We can't directly set $\nabla_w L = 0$, so we need an iterative algorithm. At step $t$, we can update
$$ w^{(t+1)}=w^{(t)}+\eta\nabla_wL \ \ \nabla_wL = \sum_{i=1}{n}(1-\sigma_i(y_i\cdot w))y_ix_i $$


#### Algorithm

**Input**: Training data $(x_1, y_1),...,(x_n, y_n)$ and step size $\eta > 0$
1. **Set** $w^{(1)} = \overrightarrow{0}$
2. **For step** $t = 1,2,...$ **do**
    - Update $w^{(t+1)} = w^{(t)} + \eta\sum_{i=1}^{n}(1-\sigma_i(y_i\cdot w))y_ix_i$

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
%%time
# Build logistic model with L2 regularization
param_grid = {'penalty': ['l1', 'l2'],
              'C': [0.1, 1, 10, 100], 
              'tol': [1e-4, 1e-5], 
              'max_iter': [200, 500]}
gs_lr = GridSearchCV(LogisticRegression(), param_grid, cv=5)
gs_lr.fit(X_train, y_train)
print("Best Parameters:", gs_lr.best_params_)
print("Accuracy on Training Set:", gs_lr.best_score_)

y_pred_prob = gs_lr.predict_proba(X_test)[:,1]
print("Accuracy on Test Set:", gs_lr.score(X_test, y_test))
print("AUC:", roc_auc_score(y_test, y_pred_prob))

In [ ]:
y_pred = gs_lr.predict(X_test)
print(y_pred_prob[:5])
print(y_pred[:5])
print(y_test[:5])

In [ ]:
# Moving the threshold does not improve prediction
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    y_pred_rev = [1 if i > t else 0 for i in y_pred_prob]
    print('Threshold: {}, Accuracy: {}'.format(t, accuracy_score(y_pred_rev, y_test)))

### 4. Support Vector Machine

With two linearly separable classes, choose a hyperplane such that its distance to the **closest point in each class** is maximized to achieve good generalization (low prediction error).

#### Convex Sets and Convex Hulls
Where a seperating hyperplane may be placed depends on the "outer" points on the sets. Points in the center do not matter. In geometric terms, we can represent each class by the smallest convex set which contains all point in the class. This is called a $\textit{convex hull}$.

A convex hull is defined by all possible weighted averages of points in a set. That is, let $x_1,...x_n$ be the data coordinates. Every point $x_0$ in the convex hull can be reached by setting
$$ x_0 = \sum_{i=1}^{n}\alpha_ix_i, \ \ \alpha_i \geq0, \ \ \sum_{i=1}^{n}\alpha_i = 1,$$
for some $(\alpha_1,...\alpha_n)$. No point outside the convex hull can be reached this way.

#### Algorithm
For $n$ seperate points $(x_1,y_1),...,(x_n,y_n)$ with $y_i \in {\pm1}$, solve:
$$ \min_{w, w_0} \frac{1}{2}\|w\|^2 $$
subject to
$$ y_i(x_i^Tw + w_0) \geq 1 \ \ for\ i = 1,...,n$$

- If there exists a hyperplan $H$ that separates the classes, we can scale $w$ so that $y_i(x_i^Tw+w_0)>1$ for all $i$.
- This formula only has a solution when the classes are linearly separable.

Solving above foluma would require $\textit{Lagrange multipliers}$. After derived with $\textit{Lagrange multipliers}$, the formula will eventually turn into:
$$\min_{\alpha_1,...\alpha_n}\bigg|\bigg(\sum_{i\in S_1}\frac{\alpha_i}{C}x_i\bigg) - 
\bigg(\sum_{j\in S_0}\frac{\alpha_j}{C}x_j\bigg)\bigg|^2,$$
where 
- $S_i$ and $S_0$ are the sets of $x$ in class $+1$ and $-1$
- $C:=\sum_{i\in S_1}\alpha_u = \sum_{j\in S_0}\alpha_j,\ \ \alpha_i\geq0$

Therefore, the algorithm is to find the closest points in the convex hulls constructed from the data in class $+1$ and $-1$.

#### Soft-Margin SVM

If the data isn't linearly separable, permit training data be on wrong side of hyperplane at a cost by replacing the training rule $y_i(x_i^Tw + w_0) \geq 1$  with 
$$y_i(x_i^Tw+w_0)\geq 1 - \xi_i, \ \ with \ \ \xi_i \geq0.$$

The $\xi_i$ are also called $\textit{slack variables}.$

The function therefore becomes:
$$ \min_{w, w_0,\xi_1,...\xi_n} \frac{1}{2}\|w\|^2 + \lambda\sum_{i=1}^{n}\xi_i$$
subject to
$$ y_i(x_i^Tw + w_0) \geq 1 - \xi_i \ \ for\ i = 1,...,n$$
$$ \xi_i \geq 0 \ \ for \ \ i = 1,...,n$$

- If $\lambda$ is very small, we're happy to misclassify.
- For $\lambda \rightarrow \infty$, we recover the original SVM because we want $\xi_i=0$.
- We can use cross-valudation to choose $\lambda$

In [ ]:
from sklearn.svm import SVC
from sklearn.svm import LinearSVC

In [ ]:
%%time
#l2 penalty
param_grid = {'tol': [1e-4, 1e-5, 1e-6],
              'C': [0.1, 1, 10],
              #'max_iter': [500, 1000, 1500]
             }
gs_lsvc = GridSearchCV(SVC(kernel ='linear', probability=True), param_grid, cv=5)
gs_lsvc.fit(X_train, y_train)
print("Best Parameters:", gs_lsvc.best_params_)
print("Accuracy on Training Set:", gs_lsvc.best_score_)

y_pred_prob = gs_lsvc.predict_proba(X_test)[:,1]
print("Accuracy on Test Set:", gs_lsvc.score(X_test, y_test))
print("AUC:", roc_auc_score(y_test, y_pred_prob))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

log_reg = LogisticRegression(solver='liblinear', penalty='l1')
param_grid = {'C': [0.1, 1, 10], 'max_iter': [100, 200, 300]}
grid_search = GridSearchCV(log_reg, param_grid, cv=5)
grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Accuracy on Training Set:", grid_search.best_score_)

# En iyi model ile test seti üzerindeki doğruluğu hesapla
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)

print("Accuracy on Test Set:", test_accuracy)


### 5. Kernelizing SVM

#### Kernel
A kernel $K(\cdot,\cdot) : \mathbb{R}^d \times \mathbb{R}^d \rightarrow \mathbb{R}$ is a symmetric function defined as follows:

For any set of $n$ data points $x_1,...,x_n \in \mathbb{R}^d$, the $n\times n$ matrix $K$, where $K_{ij} = K(x_i, x_j)$, is $\textit{positive semidefinite}$. (Note: The output of the kernel function is greater or equal to 0.)

Intuitively, this means $K$ satisfies the properties of a covariance matrix.

#### Mercer's theorem
If the function $K(\cdot,\cdot)$ satisfies the above properties, then there exists a mapping $\phi:\mathbb{R}^d\rightarrow\mathbb{R}^D$ such that
$$k(x_i, x_j) = \phi(x_i)^T\phi(x_j).$$

(Note: If this is satisfied for every single set of n vectors in $\mathbb{R}^d$ where $n$ is arbitrary and the vectors themselves are arbitrary, **then there exists some function** $\phi$, which is a function that takes in any particular $x$ and it performs the same function on that particular $x$ to map it to another space. And we can then have this dot product representation of the kernel function.)

If we first define $\phi(\cdot)$, the mapping function, and then $K$, then this is obvious. However, sometimes we first define $K(\cdot,\cdot)$ and avoid ever using $\phi(\cdot)$.

(Note: We use kernel to get results of the dot products of two $x_i$ and $x_j$ mapped to a higher place without actually mapping the $x_i$ and $x_j$ and calculating the dot product.)

#### RBF
By far the most popular kernel is the Gaussian kernel, also called the radiial basis function (RBF),
$$ K(x, x') = \alpha \ exp\big\{ - \frac{1}{b}\|x-x'\|^2 \big\}. $$

- It takes into account proximity in $\mathbb{R}^d$. Things close together in space have larger value (as defined by kernel width $b$).

In this case, the mapping $\phi(x)$ that produces the RBF kernel is $\textit{infinite dimensional}$ (it's a continuous function instead of a vector). Therefore
$$K(x,x') = \int\phi_t(x)\phi_t(x')dt.$$

(Note: there's a function of $x$ and $t$ such that this kernel results by integratign the product of those two functions.)
- $K(x,x')$ is like a Gaussian on $x$ with $x'$ as the mean (or vice versa).

#### Algorithm
Map the data into higher dimension using the function $\phi(x_i)$,
$$ \min_{w, w_0,\xi_1,...\xi_n} \frac{1}{2}\|w\|^2 + \lambda\sum_{i=1}^{n}\xi_i$$
subject to
$$ y_i(\phi(x_i)^Tw + w_0) \geq 1 - \xi_i \ \ for\ i = 1,...,n$$
$$ \xi_i \geq 0 \ \ for \ \ i = 1,...,n$$

To classify a new point:
$$ y_0 = sign\big(\sum_{i=1}^{n}\alpha_iy_i\phi(x_0)T\phi(x_i)+w_0)\big)=
sign(\sum_{i=1}^{n}\alpha_iy_iK(x_0,x_i)+w_0\big)$$

- We're still learning a linear classifier, in the higher dimensional map space, but when we look at what the decision is in the original space, we get non-linear decision boundaries.
- In practice, we choose a kernel function (e.g., RBF) and use cross-validation for $\lambda$ parameter and RBF kernel width.

In [ ]:
%%time
param_grid = {'C': [0.01, 0.1, 1, 10, 100],
              'gamma': [1e-2, 1/753, 1e-4, 1e-5],
              'tol': [1e-3, 1e-4, 1e-5],
              #'max_iter': [500, 1000]
             }
gs_svc = GridSearchCV(SVC(probability=True), param_grid, cv=5)
gs_svc.fit(X_train, y_train)
print("Best Parameters:", gs_svc.best_params_)
print("Accuracy on Training Set:", gs_svc.best_score_)

y_pred_prob = gs_svc.predict_proba(X_test)[:,1]
print("Accuracy on Test Set:", gs_svc.score(X_test, y_test))
print("AUC:", roc_auc_score(y_test, y_pred_prob))

So far support vector machine with kernel generates the best results.

### 6. Decision Tree
A decision tree maps input $x \in \mathbb{R}^d$ to output $y$ using binary decision rules:
- Each node in the tree has a splitting rule.
- Each leaf node is associated with an output value (outputs can repeat)

Each splitting rule is of the form $h(x) = \mathbb{I}\{x_j>t\}$ for some dimension $j$ of $x$ and $t\in\mathbb{R}$. Using these transition rules, a path to a leaf node gives the prediction.

The basic method for learning tree is with a top-down greedy algorithm. Measure of quality of prediction include
1. Classificataion error: $1 - \max_kp_k$
2. Gini index: $1 - \sum_{k}p_k^2$
3. Entrophy: $-\sum_{k}p_k\ln p_k$

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
%%time
param_grid = {'criterion': ['gini', 'entropy'],
              'max_depth': [3, 5, 8, None],
              'min_samples_split': [2, 3, 5]}
gs_dt = GridSearchCV(DecisionTreeClassifier(), param_grid, cv=5)
gs_dt.fit(X_train, y_train)
print("Best Parameters:", gs_dt.best_params_)
print("Accuracy on Training Set:", gs_dt.best_score_)

y_pred_prob = gs_dt.predict_proba(X_test)[:,1]
print("Accuracy on Test Set:", gs_dt.score(X_test, y_test))
print("AUC:", roc_auc_score(y_test, y_pred_prob))

## III. Model Improvement

Now there are 753 features, including 752 numeric features, in the dataset. Considering the massive number of features, using dimension reduction method to reduce features and leave only more discriminative ones would improve modeling speed and potentially prediction performance.

### PCA
Principle component analysis is often used for dimension reduction. Imagine we're given data in $R^d$, where $d$ is very high-dimensional, and we don't think that the information fills all of those dimensions. We can **take all the information contained in that high-dimensional space and map it to a much lower-dimensional space, where we don't lose much information**. 

#### Algorithm - The First Principal Component
This is related to the problem of finding the largest eigenvalue,
$$q = \operatorname*{arg\,min}_{q} \sum_{i=1}^{n}\|x_i - qq^Tx_i\|^2 \ \ \ \  s.t.\ \ q^Tq = 1 $$
$$ = \operatorname*{arg\,min}_{q} \sum_{i=1}^{n}x_i^Tx_i - q^T\big(\sum_{i=1}^{n}x_ix_i^T\big)q$$
$$\sum_{i=1}^{n}x_ix_i^T = XX^T$$

We define $X = [x_1,...,x_n]$. Since the first term doesn't depend on $q$ and we have a negative sing in front of the second term, equivalently we solve 
$$q = \operatorname*{arg\,max}_{q} q^T(XX^T)q \ \ \ \ subject to \ \ q^Tq = 1$$

This is the eigendecomposition problem:
- $q$ is the first eigenvector of $XX^T$
- $\lambda = q^T(XX^T)q$ is the first eigenvalue


#### Algorithm - General
The general form of PCA considers $K$ eigenvectors,
$$q = \operatorname*{arg\,min}_{q} \sum_{i=1}^{n}\|x_i - \sum_{k=1}^{K}(x_i^Tq_k)q_k\|^2 \ \ \ \  s.t.\ \ q^Tq = 1, k = k'; q^Tq = 0, k \neq k' $$
$$ = \operatorname*{arg\,min}_{q} \sum_{i=1}^{n}x_i^Tx_i - \sum_{k=1}^{K}q_k^T \big(\sum_{i=1}^{n}x_ix_i^T\big)q_k$$
$$\sum_{i=1}^{n}x_ix_i^T = XX^T$$

The vectors in $Q = [q_i,...,q_k]$ give us a $K$ dimensional subspace with which to represent the data:
$$x_{proj} = \begin{bmatrix}
               q_1^Tx \\
               \vdots \\
               q_K^Tx
             \end{bmatrix}, x \approx \sum_{k=1}^{K}(q_k^Tx)q_k = Qx_{proj}$$
             
The eigenvectors of $(XX^T)$ can be learned as below.

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
X = df.groupby('id').mean().drop(['class'], axis=1)
y = df[['id', 'class']].groupby('id').mean()['class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=91)

In [ ]:
scaler = MinMaxScaler()
pca = PCA()
pca.fit(scaler.fit_transform(X))

In [ ]:
df_pca = pd.DataFrame({'variance_ratio': pca.explained_variance_ratio_})
df_pca['cumulated_ratio'] = df_pca['variance_ratio'].cumsum()
df_pca.head()

In [ ]:
plt.bar(df_pca.index[:100], df_pca['variance_ratio'][:100])
plt.show()

In [ ]:
plt.bar(df_pca.index[:100], df_pca['cumulated_ratio'][:100])
plt.show()

In [ ]:
for i in [0.8, 0.85, 0.9, 0.95]:
    print('Top {} features explains {} of the variance.'.format(df_pca[df_pca['cumulated_ratio'] > i].index[0], i))

### Transformer
Build a transformer that apply PCA on numerical columns and convert gender to catetorical problem.

***Note 1: Tried to build transformer but it doesn't work in pipeline. As the test set in GridSearchCV would regenerate a smaller n_component that is equal to the sample size of test set.***


***Note 2: Tried to exclude gender to do PCA and run models in Model Selection section. However, the result is not better, and sometimes even worse, than the results of running PCA with gender included.***

Therefore, in the following pipelines, no specific transformer is needed as gender will be processed as a numerical variable in scaler, PCA, and modeling.

In [ ]:
# from sklearn.base import BaseEstimator, TransformerMixin

In [ ]:
# class PCAexclGender(BaseEstimator, TransformerMixin):
    
#     def __init__( self, n_components=10 ):
#         self.n_components = n_components
    
#     def fit(self, X, y=None):
#         return self
    
#     def transform(self, X, y=None):
#         X_withoutGender = X.drop(['gender'], axis=1)
#         scaler=MinMaxScaler()
#         pca=PCA(n_components=self.n_components)
#         X_pca = pd.DataFrame(pca.fit_transform(scaler.fit_transform(X_withoutGender)))
#         X_pca['gender'] = pd.Series(X.gender.values).astype('category')
        
#         return X_pca

In [ ]:
# scaler_test = PCAexclGender()
# scaler_test.fit(X_train)
# scaler_test.transform(X_train)
# #scaler_test
# #X_scaler_test = scaler_test.transform(X_test)
# #X_scaler_test

### Pipeline
Use pipeline on standard scaler, PCA, and SVC, the best model so far, to see if there's any improvement.

In [ ]:
from sklearn.pipeline import Pipeline

In [ ]:
%%time
pipeline = Pipeline([('Scaler', MinMaxScaler()),
                     ('PCA', PCA()),
                     ('SVC', SVC(probability=True))])
parameters = {'PCA__n_components': [80, 100, 120, 150],
              'SVC__C': [0.01, 0.1, 1, 10, 100],
              'SVC__gamma': [1e-2, 1e-3, 1e-4, 1e-5],
              'SVC__tol': [1e-3, 1e-4, 1e-5],
              'SVC__max_iter': [500, -1]
             }
gs_pca_svc = GridSearchCV(pipeline, param_grid=parameters, cv=5)
gs_pca_svc.fit(X_train, y_train)
print("Best Parameters:", gs_pca_svc.best_params_)
print("Accuracy on Training Set:", gs_pca_svc.best_score_)

y_pred_prob = gs_pca_svc.predict_proba(X_test)[:,1]
print("Accuracy on Test Set:", gs_pca_svc.score(X_test, y_test))
print("AUC:", roc_auc_score(y_test, y_pred_prob))

After adding new features and applying PCA, the accuracy on test set improved. See next cell for improvements on other models.

## IV. Model Selection

Apply pipeline on all models to compare the best performance on accuracy and AUC with a narrow-lown list.

***Note: Some models with best parameters roughly identified through plentiful times of running would have a shorter list of parameters to search.***

In [ ]:
%%time
warnings.simplefilter('ignore')
kNN = GridSearchCV(Pipeline([('Scaler', MinMaxScaler()), 
                             ('PCA', PCA()), ('kNN', KNeighborsClassifier())]),
                   param_grid={'PCA__n_components': [80, 100],
                               'kNN__n_neighbors': np.arange(3, 6)}, cv=5)

NB = GridSearchCV(Pipeline([('scaler', MinMaxScaler()), ('PCA', PCA()), ('NB', GaussianNB())]),
                  param_grid={'PCA__n_components': [80, 100]}, cv=5)

LR = GridSearchCV(Pipeline([('Scaler', MinMaxScaler()), 
                            ('PCA', PCA()), ('LR', LogisticRegression())]),
                  param_grid={'PCA__n_components': [80, 100],
                              'LR__penalty': ['l1', 'l2'],
                              'LR__C': [0.01, 0.1, 1],
                              'LR__tol': [1e-3, 1e-4, 1e-5], 
                              #'LR__max_iter': [100, 150]
                             }, cv=5)

SVM2 = GridSearchCV(Pipeline([('Scaler', MinMaxScaler()), 
                              ('PCA', PCA()), ('SVC', SVC(kernel='linear', probability=True))]),
                    param_grid={'PCA__n_components': [80, 100],
                                'SVC__tol': [1e-4],
                                'SVC__C': [0.01, 0.1, 1],
                                #'SVC__max_iter': [500, 1000, 1500]
                               }, cv=5)

SVM1 = GridSearchCV(Pipeline([('Scaler', MinMaxScaler()), ('PCA', PCA()), 
                              ('SVC', LinearSVC(loss='l2', penalty='l1', dual=False))]),
                    param_grid={'PCA__n_components': [80, 100],
                                'SVC__C': [0.1, 1, 10]}, cv=5)

SVM = GridSearchCV(Pipeline([('Scaler', MinMaxScaler()), ('PCA', PCA()),
                             ('SVC', SVC(probability=True))]),
                   param_grid={'PCA__n_components': [80, 100],
                               'SVC__C': [1, 10, 100],
                               'SVC__gamma': [1e-2, 1e-3],
                               'SVC__tol': [1e-3],
                               'SVC__max_iter': [500]
                              }, cv=5)

DT = GridSearchCV(Pipeline([('Scaler', MinMaxScaler()), 
                            ('PCA', PCA()), ('DT', DecisionTreeClassifier())]),
                  param_grid={'PCA__n_components': [80, 100],
                              'DT__criterion': ['gini', 'entropy'],
                              'DT__max_depth': [3, 5, 8, None],
                              'DT__min_samples_split': [2, 3, 5]}, cv=5)

models = {'Naive Bayes': NB, 'Logistic Regression': LR, 
          'SVM-L2': SVM2, 'SVM-L1': SVM1, 'SVM': SVM, 'Decision Tree': DT}

for k in models:
    models[k].fit(X_train, y_train)
    print("Best Parameters of {}:".format(k), models[k].best_params_)
    print("Accuracy on Training Set:", models[k].best_score_)
    print("Accuracy on Test Set:", models[k].score(X_test, y_test))
    if k not in ['SVM-L1']:
        print("AUC:", roc_auc_score(y_test, models[k].predict_proba(X_test)[:,1]))
    print("\n")

In [ ]:
# %%time
# LR = GridSearchCV(Pipeline([('Scaler', MinMaxScaler()), 
#                             ('PCA', PCA()), ('LR', LogisticRegression())]),
#                   param_grid={'PCA__n_components': [80, 100, 120],
#                               'LR__penalty': ['l1', 'l2'],
#                               'LR__C': np.linspace(0.05, 0.25, 15),
#                               'LR__tol': [1/80, 0.01, 0.05, 1e-3, 1e-4, 1e-5, 1e-6], 
#                               #'LR__max_iter': [100, 150]
#                              }, cv=5)
# LR.fit(X_train, y_train)
# print("Best Parameters:", LR.best_params_)
# print("Accuracy on Training Set:", LR.best_score_)

# y_pred_prob = LR.predict_proba(X_test)[:,1]
# print("Accuracy on Test Set:", LR.score(X_test, y_test))
# print("AUC:", roc_auc_score(y_test, y_pred_prob))

In [ ]:
# np.linspace(0.05, 0.25, 15)

Based on iterative parameter tuning, the three best models that have potential to generate high accuracy and AUC are linear SVM with L2 penalty, SVM with kernel, and logistic regression.

### ROC Curve
Here only the top three best models are focused.

In [ ]:
from sklearn.metrics import roc_curve
from sklearn.metrics import classification_report

In [ ]:
y_pred_prob_lr = LR.predict_proba(X_test)[:,1]
y_pred_prob_lsvm = SVM2.predict_proba(X_test)[:,1]
y_pred_prob_svm = SVM.predict_proba(X_test)[:,1]
fpr_lr, tpr_lr, thresholds_lr = roc_curve(y_test, y_pred_prob_lr)
fpr_lsvm, tpr_lsvm, thresholds_lsvm = roc_curve(y_test, y_pred_prob_lsvm)
fpr_svm, tpr_svm, thresholds_svm = roc_curve(y_test, y_pred_prob_svm)
plt.plot([0,1], [0,1], 'k--')
plt.plot(fpr_lr, tpr_lr, alpha=0.5, label='Logistic Regression')
plt.plot(fpr_lsvm, tpr_lsvm, alpha=0.5, label='Support Vector Machine - Linear')
plt.plot(fpr_svm, tpr_svm, alpha=0.5, label='Support Vector Machine - Kernel')
plt.legend()
plt.show()

The three models have roughly the same area under ROC curve.

### Change Threshold

In [ ]:
print("Logistic Regression")
for t in [0.4, 0.45, 0.5, 0.55, 0.6]:
    y_pred_rev = [1 if i > t else 0 for i in y_pred_prob_lr]
    print('Threshold: {}, Accuracy: {}'.format(t, accuracy_score(y_pred_rev, y_test)))

In [ ]:
print("Support Vector Machine - Linear")
for t in [0.4, 0.45, 0.5, 0.55, 0.6]:
    y_pred_rev = [1 if i > t else 0 for i in y_pred_prob_lsvm]
    print('Threshold: {}, Accuracy: {}'.format(t, accuracy_score(y_pred_rev, y_test)))

Lowering the threshold of logistic regression and linear SVM, we can improve the accuracy for around 0.02 respectively.

In [ ]:
print("Support Vector Machine - Kernel")
for t in [0.4, 0.45, 0.5, 0.55, 0.6]:
    y_pred_rev = [1 if i > t else 0 for i in y_pred_prob_svm]
    print('Threshold: {}, Accuracy: {}'.format(t, accuracy_score(y_pred_rev, y_test)))

Changing threshold doesn't help improve the accuracy of SVM with kernel.

### Classification Report

In [ ]:
print("Logistic Regression:")
y_pred_rev = [1 if i > 0.45 else 0 for i in y_pred_prob_lr]
print(classification_report(y_test, y_pred_rev))
print("AUC: {}".format(roc_auc_score(y_test, y_pred_prob_lr)))

In [ ]:
print("Support Vector Machine - Linear:")
y_pred_rev = [1 if i > 0.45 else 0 for i in y_pred_prob_lsvm]
print(classification_report(y_test, y_pred_rev))
print("AUC: {}".format(roc_auc_score(y_test, y_pred_prob_lsvm)))

In [ ]:
print("Support Vector Machine - Kernel:")
print(classification_report(y_test, SVM.predict(X_test)))
print("AUC: {}".format(roc_auc_score(y_test, y_pred_prob_svm)))

### Best Model
The Best model is the pipeline that combines min-max normalization, PCA, and linear SVM with L2 penalty. The best parameters of PCA and linear SVM is as below. After applying the following model and lower the threshold to 0.45 for prediction, we get the accuracy of 0.902 and AUC of 0.905.

In [ ]:
# {'PCA__n_components': 80, 'SVC__C': 0.1, 'SVC__tol': 0.0001}
SVM2.best_params_

In [ ]:
# Second best models
# {'SVC__max_iter': 500, 'PCA__n_components': 80, 'SVC__tol': 0.001, 'SVC__gamma': 0.01, 'SVC__C': 10}
print(SVM.best_params_)
# {'LR__C': 0.1, 'PCA__n_components': 100, 'LR__tol': 0.001, 'LR__penalty': 'l2'}
print(LR.best_params_)

## V. Appendix

### Appendix A: Concepts for Logistic Regression:

#### A1. Binay Classification Type
Input $x_i \in \mathbb{R}^b$ and output $y_i \in {\pm1}$

we define a $\textit{classifier f}$, which makes prediction $y_i = f(x_i, \Theta)$ based on a function of $x_i$ and parameters $\Theta$. In other words $f: \mathbb{R}^d \rightarrow {-1, +1}$

In **Bayes classificaiton** framework, $\Theta$ contains:
1.  class prior probabilities on $y$,
2. parameters for calss-dependent distribution on $x$.

In **linear classification** framework, the prediction is linear in the parameters $\Theta$.

**Bayes classification** and **linear classification** are connected through ***log odds***.

#### A2. Log Odds
With Bayes classifier, we declare class $y=1$ if
$$ p(x|y = 1)P(y = 1) > p(x|y = 0)P(y = 0) $$

$$ \Updownarrow $$

$$ \ln\frac{p(x|y = 1)P(y = 1)}{p(x|y = 0)P(y = 0)} > 0 $$

The second line is referred to as the $\textit{log odds}$.

#### A3. Lineaer Discriminant Analysis
In the case where $p(x|y) = N(x| \mu_y, \Sigma)$ **(a single Gaussian with a shared covariance matrix)**

$$ \ln\frac{p(x|y = 1)P(y = 1)}{p(x|y = 0)P(y = 0)} =
\ln\frac{\pi_1}{\pi_0} - \frac{1}{2}(\mu_0 + \mu_1)^T\Sigma^{-1}(\mu_1 - \mu_0) + x^T\Sigma^{-1}(\mu_1 - \mu_0)$$

This is also called ***lineaer discriminant analysis*** (used to be called LDA).

So we ca write the decision rule for the Bayes classifer as a linear one:
$$ f(x) = sign(x^Tw + w_0) $$
where

$$ w_0 = \ln\frac{\pi_1}{\pi_0} - \frac{1}{2}(\mu_0 + \mu_1)^T\Sigma^{-1}(\mu_1 - \mu_0) $$
$$ w = \Sigma^{-1}(\mu_1 - \mu_0) $$

This Bayes classifier is one instance of a linear classifier.

Setting $w_0$ and $w$ this way may be too restrictive - it assumes single Gaussian with shared covariance. If we relax what values $w_0$ and $w$ can take we can do better. 

### Appendix B: Linear Classifiers

#### B1. Definition
A $\textit{binary linear classifier}$ is a function of the form
$$f(x) = sing(X^Tw + w_0),$$
where $w \in \mathbb{R}^d$ and $w_0 \in \mathbb{R}$. Since the goal is to learn $w, w_0$ from the data, we are assuming that $\textit{linear separability}$ in $x$ is an accurate property of the classes.

#### B2. Linear Separability
Two sets $A, B \subset \mathbb{R}^d$ arfe called linearly separable if

$$x^Tw + w_0 > 0 \ \ if \ \ x \in A (e.g, class +1)$$
$$x^Tw + w_0 < 0 \ \ if \ \ x \in B (e.g, class -1)$$

The pair $(w, w_0)$ defines an $\textit{affine hyperplane}$. It is important to develop the right geometric understanding about what this is doing.

#### B3. Two methods:

- **Least squares:** One simple idea is to treat classification as a regression problem. However, using regression for classification problem is not robust because it's sensitive to outliers
- **Perceptron:** The perceptron represents a first attempt at linear classification by directly learning the hyper plane defined by $w$. It is not used as much anymore because of some drawbacks: convergence issues and the assumption on linear seperability.

## References for Model Introduction and Algorithms
- Applied Machine Learning Certification - Columnbia Engineering Executive Education
- Post Graduate Diploma of Applied Machine Learning and Artificial Intelligence - Columnbia Engineering Executive Education

#### Note: The coding was done through personal works and researches and not borrowed from the certification course.